# 🇪🇹 Tibeb AI — GPU Fine-Tuning on Colab

**Requirements:** Colab Pro (A100 GPU), Google Drive for data

**What this does:**
1. Installs dependencies
2. Loads training data from Google Drive
3. Fine-tunes Aya 8B with QLoRA on A100
4. Saves adapter to Drive

**Estimated time:** 2-3 hours on A100, 4-6 hours on L4/T4

## Step 0: Connect GPU & Mount Drive
Make sure you selected **A100 GPU** in Runtime → Change runtime type

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create project directory
import os
PROJECT_DIR = '/content/drive/MyDrive/tibeb-training'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
print(f'Project dir: {PROJECT_DIR}')

## Step 1: Install Dependencies

In [ ]:
!pip install -q "transformers>=4.44.0" "datasets>=2.18.0" "peft>=0.10.0" "trl>=0.8.0" \
  "accelerate>=0.28.0" "bitsandbytes>=0.43.0" safetensors huggingface_hub wandb
# flash-attn is optional — skip if it fails
!pip install -q flash-attn --no-build-isolation 2>/dev/null || echo "flash-attn skipped (will use default attention)"

## Step 2: Upload Training Data

**Option A:** Upload `tibeb_unified_train.jsonl` from your Mac to Google Drive → `tibeb-training/data/`

**Option B:** Upload directly to Colab (temporary — lost on disconnect):

In [ ]:
# Find training data — checks multiple locations
DATA_PATH = None
search_paths = [
    f'{PROJECT_DIR}/tibeb_unified_train.jsonl',          # tibeb-training/tibeb_unified_train.jsonl
    f'{PROJECT_DIR}/data/tibeb_unified_train.jsonl',     # tibeb-training/data/
    '/content/tibeb_unified_train.jsonl',                 # uploaded directly to Colab
]
for p in search_paths:
    if os.path.exists(p):
        DATA_PATH = p
        print(f'Found data: {p}')
        !wc -l {p}
        break

if DATA_PATH is None:
    from google.colab import files
    print('Data not found in Drive. Upload tibeb_unified_train.jsonl:')
    uploaded = files.upload()
    DATA_PATH = '/content/tibeb_unified_train.jsonl'

In [ ]:
# Verify data
import json

rows = 0
sources = {}
with open(DATA_PATH, encoding='utf-8') as f:
    for line in f:
        row = json.loads(line.strip())
        src = row.get('source', 'unknown')
        sources[src] = sources.get(src, 0) + 1
        rows += 1

print(f'Total rows: {rows:,}')
print(f'\nSources:')
for s, c in sorted(sources.items(), key=lambda x: -x[1]):
    print(f'  {s:30s} {c:>7,} ({c/rows*100:.1f}%)')

## Step 3: Load Model & Configure Training

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

MODEL_ID = 'CohereForAI/aya-expanse-8b'
OUTPUT_DIR = f'{PROJECT_DIR}/models/tibeb-sft-gpu'

TIBEB_SYSTEM_PROMPT = (
    "ቲብብ ነኝ — የኢትዮጵያ የፋይናንስ AI ረዳት።"
    " ስለ ኢንቨስትመንት፣ ቁጠባ፣ አክሲዮን ገበያ፣ ቲ-ቢል፣ እና የገንዘብ ጉዳዮች በአማርኛ እረዳለሁ።"
    " ሁልጊዜ አክብሮት ያለው፣ ትምህርታዊ፣ እና ግልጽ መልስ እሰጣለሁ።"
    " ቀጥተኛ የኢንቨስትመንት ምክር አልሰጥም — ተገቢውን ባለሙያ እንዲያማክሩ እመክራለሁ።"
)

FINANCIAL_SOURCES = {'tibeb_financial', 'tibeb_financial_v2', 'sujet_finance',
                     'tibeb_safety', 'tibeb_voice', 'tibeb_deep'}

print(f'Model: {MODEL_ID}')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
# Load and format data
def format_row(row):
    instruction = row.get('instruction', '')
    inp = (row.get('input') or '').strip()
    output = row.get('output', '')
    source = row.get('source', '')

    user_msg = f"{instruction}\n\n{inp}" if inp else instruction

    messages = []
    if source in FINANCIAL_SOURCES:
        messages.append({'role': 'system', 'content': TIBEB_SYSTEM_PROMPT})
    messages.append({'role': 'user', 'content': user_msg})
    messages.append({'role': 'assistant', 'content': output})
    return {'messages': messages}

print('Loading data...')
rows = []
skipped = {'short': 0, 'empty': 0, 'echo': 0}
with open(DATA_PATH, encoding='utf-8') as f:
    for line in f:
        row = json.loads(line.strip())
        if not row.get('instruction') or not row.get('output'):
            skipped['empty'] += 1; continue
        output = row['output'].strip()
        if len(output) < 10:
            skipped['short'] += 1; continue
        inp = (row.get('input') or '').strip()
        if inp and output == inp:
            skipped['echo'] += 1; continue
        rows.append(format_row(row))

print(f'Loaded: {len(rows):,} rows')
print(f'Filtered: {skipped}')

# Split
import random
random.seed(42)
random.shuffle(rows)
split = int(len(rows) * 0.98)  # 98% train, 2% eval
train_data = rows[:split]
eval_data = rows[split:]
print(f'Train: {len(train_data):,}, Eval: {len(eval_data):,}')

In [ ]:
# Load tokenizer
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model in 4-bit
print('Loading model (4-bit QLoRA)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2',
)
model = prepare_model_for_kbit_training(model)
print(f'Model loaded: {model.dtype}')

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=64,                    # Higher rank than Mac version (was 16)
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],  # All linear layers
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 4: Train!

In [ ]:
# Format for SFTTrainer
def format_chat(example):
    return tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False
    )

train_ds = Dataset.from_list(train_data)
eval_ds = Dataset.from_list(eval_data)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,      # Effective batch size: 32
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.01,
    bf16=True,
    logging_steps=25,
    save_strategy='steps',
    save_steps=500,
    eval_strategy='steps',
    eval_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    max_grad_norm=1.0,
    report_to='none',
    seed=42,
    dataloader_num_workers=4,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    formatting_func=format_chat,
    max_seq_length=1024,   # Longer than Mac version (was 512)
    packing=True,          # Efficient packing of short sequences
)

print(f'Training config:')
print(f'  Epochs: {training_args.num_train_epochs}')
print(f'  Batch: {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'  LR: {training_args.learning_rate}')
print(f'  Seq length: 1024')
print(f'  LoRA rank: 64')
print(f'  Packing: True')

In [ ]:
# TRAIN
print('Starting training...\n')
trainer.train()
print('\nTraining complete!')

In [ ]:
# Save the best model
FINAL_DIR = f'{PROJECT_DIR}/models/tibeb-sft-final'
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f'Model saved to: {FINAL_DIR}')

# Also save to local Colab for quick testing
trainer.save_model('/content/tibeb-adapter')
tokenizer.save_pretrained('/content/tibeb-adapter')
print('Also saved locally for testing')

## Step 5: Test the Model

In [ ]:
# Test prompts
from peft import PeftModel

EVAL_PROMPTS = [
    ('basic', 'ቲ-ቢል ምንድነው?'),
    ('advice', 'ለጀማሪ ኢንቨስተር ምን ትመክራለህ?'),
    ('comparison', 'ቁጠባ ባንክ ውስጥ ማስቀመጥ ወይስ ኢንቨስት ማድረግ ይሻላል?'),
    ('esx', 'የኢትዮጵያ ሴኩሪቲስ ኤክስቼንጅ ምንድነው?'),
    ('telebirr', 'ቴሌብር ተጠቅሜ አክሲዮን መግዛት እችላለሁ?'),
    ('greeting', 'ሰላም! ስለ ኢትዮጵያ የካፒታል ገበያ ንገረኝ።'),
    ('risk', 'የአክሲዮን ገበያ ሪስክ ምንድነው?'),
    ('formal', 'እርስዎ ስለ ቲ-ቢል ማብራሪያ ቢሰጡኝ?'),
]

model.eval()
print('=' * 70)
print('  Tibeb Model Evaluation')
print('=' * 70)

for tag, prompt in EVAL_PROMPTS:
    messages = [
        {'role': 'system', 'content': TIBEB_SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n{"─" * 70}')
    print(f'  [{tag}] {prompt}')
    print(f'{"─" * 70}')
    print(f'  {response.strip()[:500]}')

print(f'\n{"=" * 70}')

## Step 6: Push to HuggingFace (Optional)

In [ ]:
# Uncomment and fill in to push
# from huggingface_hub import HfApi, login
# login()  # paste your HF token
# 
# REPO_ID = 'your-username/tibeb-aya-8b-sft'
# api = HfApi()
# api.create_repo(REPO_ID, repo_type='model', exist_ok=True)
# api.upload_folder(
#     folder_path=FINAL_DIR,
#     repo_id=REPO_ID,
#     repo_type='model',
# )
# print(f'Pushed to: https://huggingface.co/{REPO_ID}')